In [ ]:
# To import my files
import sys
import os

src = os.path.abspath("./src")

if src not in sys.path:
    sys.path.append(src)

In [ ]:
# Imports
from tqdm import tqdm
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

from src.model import DeepONet
from src.dataset import generate_data

In [ ]:
# Parameters
nx, nt = 100, 50
T = 0.1

# Architecture values
m, p, hidden = 100, 64, 64

# Train hyperparameters
epochs, batch_size, lr = 2500, 128, 1e-3

device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

In [ ]:
# To generate data
u0_tr, xt, y_tr = generate_data(3000, 5, 100, 100, 50, alpha=1.0, T=0.1, seed=0)
u0_te, _,  y_te = generate_data(500,  5, 100, 100, 50, alpha=1.0, T=0.1, seed=1)

# Send data to the device
u0_tr_d, y_tr_d = u0_tr.to(device), y_tr.to(device)
u0_te_d, y_te_d = u0_te.to(device), y_te.to(device)
xt_d = xt.to(device)

loader = DataLoader(TensorDataset(u0_tr_d, y_tr_d), batch_size=batch_size, shuffle=True)

In [ ]:
model = DeepONet(m, p, hidden).to(device)
mse_loss = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=(1e-2) ** (1 / epochs))

In [ ]:
@torch.no_grad()
def rel_l2(model, u0, xt, y):
    """L2 relative error, averaged over the functions."""
    model.eval()
    pred = model(u0, xt)
    err = torch.linalg.norm(pred - y, dim=1) / torch.linalg.norm(y, dim=1)
    return err.mean().item()

In [ ]:
# Train loop
history = {"train_loss": [], "test_rel_l2": []}

model.train()
for _ in tqdm(range(1, epochs + 1), desc="Train progress", unit=" Epoch"):
    running, n_batches = 0.0, 0
    for u0_b, y_b in loader:
        optimizer.zero_grad()
        
        pred = model(u0_b, xt_d)
        loss = mse_loss(pred, y_b)
        
        loss.backward()
        optimizer.step()
        
        running += loss.item()
        n_batches += 1
        
    scheduler.step()

    history["train_loss"].append(running / n_batches)
    history["test_rel_l2"].append(rel_l2(model, u0_te_d, xt_d, y_te_d))

In [ ]:
# Train curves
fig, ax = plt.subplots(1, 2, figsize=(11, 4))

ax[0].semilogy(history["train_loss"])
ax[1].semilogy(history["test_rel_l2"], color="C1")

ax[0].set(xlabel="epoch", ylabel="MSE")
ax[0].set(xlabel="epoch", ylabel="Relative L2 Error")

plt.tight_layout()

ax[0].grid(alpha=0.4)
ax[1].grid(alpha=0.4)

plt.show()

In [ ]:
model.eval()
with torch.no_grad():
    pred_te = model(u0_te_d, xt_d).cpu()
    
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Exact and predicted matrix
exact = y_te[0].reshape(nx, nt).numpy()
pred = pred_te[0].reshape(nx, nt).numpy()

# 
err = np.abs(pred - exact)
vmin, vmax = exact.min(), exact.max()
ext = [0, T, 0, 1]

im0 = axes[0].imshow(exact, origin="lower", aspect="auto", extent=ext, vmin=vmin, vmax=vmax, cmap="viridis")
im1 = axes[1].imshow(pred, origin="lower", aspect="auto", extent=ext, vmin=vmin, vmax=vmax, cmap="viridis")
im2 = axes[2].imshow(err, origin="lower", aspect="auto", extent=ext, cmap="magma")

axes[0].set_title("Exact solution")
axes[1].set_title("DeepONet solution")
axes[2].set_title("Absolute error")

for col, im in enumerate([im0, im1, im2]):
    axes[col].set_xlabel("t")
    axes[col].set_ylabel("x")
    
    fig.colorbar(im, ax=axes[col])
    
plt.tight_layout()
plt.show()